In [1]:
# PyCIAM output visualizations with xarray and cartopy
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Load the zarr store as xarray Dataset
zarr_path = Path("pyCIAM_outputs.zarr")
ds = xr.open_zarr(zarr_path)
ds

<xarray.Dataset>
Dimensions:       (adm1: 1836, case: 11, costtype: 6, scenario: 5, quantile: 1,
                   year: 96, ssp: 5, iam: 2, seg: 9568)
Coordinates:
  * adm1          (adm1) <U8 'ABW' 'AGO.11_1' 'AGO.16_1' ... 'ZNC.3_1' 'ZNC.4_1'
  * case          (case) <U12 'noAdaptation' 'protect10' ... 'optimalfixed'
  * costtype      (costtype) <U15 'wetland' 'inundation' ... 'stormPopulation'
  * iam           (iam) <U5 'IIASA' 'OECD'
  * quantile      (quantile) float64 0.5
  * scenario      (scenario) <U17 'SSP126' 'SSP245' ... 'ncc_regional_rise'
  * seg           (seg) <U9 'seg_00001' 'seg_00002' ... 'seg_99018' 'seg_99019'
  * ssp           (ssp) <U4 'SSP1' 'SSP2' 'SSP3' 'SSP4' 'SSP5'
  * year          (year) int32 2005 2006 2007 2008 2009 ... 2097 2098 2099 2100
Data variables:
    costs         (case, costtype, adm1, scenario, quantile, year, ssp, iam) float32 dask.array<chunksize=(11, 6, 9, 5, 1, 10, 5, 2), meta=np.ndarray>
    optimal_case  (seg, scenario, quantile, ssp, iam) uint8 dask.array<chunksize=(9568, 5, 1, 5, 2), meta=np.ndarray>
Attributes:
    description:                  pyCIAM run with regional rise SLR data mapp...
    planning_period_start_years:  [2005, 2010, 2020, 2030, 2040, 2050, 2060, ...
    updated:                      Thu Feb 12 16:54:07 2026

In [5]:
ds['costtype']

<xarray.DataArray 'costtype' (costtype: 6)>
array(['wetland', 'inundation', 'relocation', 'protection', 'stormCapital',
       'stormPopulation'], dtype='<U15')
Coordinates:
  * costtype  (costtype) <U15 'wetland' 'inundation' ... 'stormPopulation'
Attributes:
    description:  stormCapital: Value of capital stock lost due to ESL\nstorm...
    long_name:    Cost Category

In [21]:
ds.sel(costtype="relocation", scenario="SSP126").sum('seg')['optimal_case'].values

array([[[169033, 145306],
        [168000, 145919],
        [165215, 146681],
        [163264, 145112],
        [161380, 142694]]], dtype=uint32)

In [31]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

zarr_path = Path("pyCIAM_outputs.zarr")
ds = xr.open_zarr(zarr_path)

ssp_indices = [0, 1, 2, 4]
ssp_labels  = ["SSP1-2.6", "SSP2-4.5", "SSP3-7.0", "SSP5-8.5"]
cases = {
    "No Adaptation":   "noAdaptation",
    "Protect (100yr)": "protect100",
    "Retreat (1m)":    "retreat1",
    "Optimal":         "optimalfixed"
}

results = {}

for case_name, case_val in cases.items():
    cost_by_ssp = {}
    for idx, label in zip(ssp_indices, ssp_labels):
        c = (
            ds.costs.sel(case=case_val, year=2100)
            .isel(scenario=0, ssp=idx, iam=0, quantile=0)
            .sum(dim="costtype")
            .load()
        )
        cost_by_ssp[label] = {
            str(ds.adm1.values[j]): float(c.isel(adm1=j).values)
            for j in range(c.sizes["adm1"])
            if np.isfinite(c.isel(adm1=j).values)
        }

    all_gids = set()
    for d in cost_by_ssp.values():
        all_gids.update(d.keys())

    cost_df = pd.DataFrame({"GID_1": list(all_gids)})
    for label in ssp_labels:
        cost_df[label] = cost_df["GID_1"].map(cost_by_ssp[label]).fillna(0)

    results[case_name] = {label: cost_df[label].sum() for label in ssp_labels}

# Print results table
col_width = 22
header = f"{'SSP':<12}" + "".join(f"{c:>{col_width}}" for c in cases.keys())
print(header)
print("-" * (12 + col_width * len(cases)))
for label in ssp_labels:
    row = f"{label:<12}" + "".join(f"${results[c][label]:>{col_width-1},.2f}" for c in cases.keys())
    print(row)

SSP                  No Adaptation       Protect (100yr)          Retreat (1m)               Optimal
----------------------------------------------------------------------------------------------------
SSP1-2.6    $ 9,824,229,984,191.25$ 3,784,719,349,718.68$ 9,924,707,590,870.14$   838,600,852,767.55
SSP2-4.5    $10,891,725,459,551.45$ 3,835,906,794,960.23$10,955,557,876,665.17$   818,773,859,927.69
SSP3-7.0    $10,406,433,405,383.75$ 3,665,383,670,664.45$10,407,859,801,678.43$   615,113,987,048.98
SSP5-8.5    $13,517,585,357,870.92$ 4,238,548,270,010.95$13,687,862,650,218.16$ 1,187,487,020,235.45


In [40]:
ssp_indices = [0, 1, 2, 4]
ssp_labels  = ["SSP1-2.6", "SSP2-4.5", "SSP3-7.0", "SSP5-8.5"]
costtypes   = ['wetland', 'inundation', 'relocation', 'protection', 'stormCapital', 'stormPopulation']

results_by_cost = {}

for idx, label in zip(ssp_indices, ssp_labels):
    cost_by_type = {}
    for ct in costtypes:
        c = (
            ds.costs.sel(case='optimalfixed', costtype=ct, year=2100)
            .isel(scenario=0, ssp=idx, iam=0, quantile=0)
            .sum(dim='adm1')
            .load()
        )
        cost_by_type[ct] = float(c.values)
    results_by_cost[label] = cost_by_type

# Build dataframe
df_costs = pd.DataFrame(results_by_cost).T
df_costs.columns = costtypes
df_costs['Total'] = df_costs.sum(axis=1)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print(df_costs.to_string())

                   wetland         inundation         relocation         protection      stormCapital   stormPopulation                Total
SSP1-2.6 49,756,958,720.00 116,456,816,640.00 174,599,618,560.00 393,147,023,360.00 43,869,257,728.00 60,771,131,392.00   838,600,806,400.00
SSP2-4.5 46,256,918,528.00 112,821,051,392.00 174,142,816,256.00 376,713,150,464.00 44,262,625,280.00 64,577,290,240.00   818,773,852,160.00
SSP3-7.0 30,271,539,200.00  77,468,622,848.00 113,941,692,416.00 307,899,596,800.00 31,729,014,784.00 53,803,540,480.00   615,114,006,528.00
SSP5-8.5 79,841,943,552.00 173,404,192,768.00 281,992,462,336.00 509,314,465,792.00 56,984,109,056.00 85,949,833,216.00 1,187,487,006,720.00
